In [ ]:
# 1 - Download, preprocess and split the dataset if necessary. Then create a dataloader for each split

from datasets import load_dataset
import torch
from torch.utils.data import Subset
import random
from dataset import ImageDataset
from degradation import ImageDegradation, DegradationParameters
from utils import plot # type: ignore
from config import TRAIN_SIZE, VALIDATION_SIZE, TEST_SIZE
from models import NAFNet, NAFNetModel

# Load ImageNet 256x256 from HuggingFace
ds = load_dataset(
    "benjamin-paine/imagenet-1k-256x256"
)

# Apply preprocessing through the custom Dataset class
# Apply preprocessing through the custom Dataset class
train_dataset = Subset(
    ImageDataset(ds["train"]),
    range(TRAIN_SIZE),
)

validation_dataset = Subset(
    ImageDataset(ds["validation"]),
    range(VALIDATION_SIZE),
)

test_dataset = Subset(
    ImageDataset(ds["test"]),
    range(TEST_SIZE),
)

naf = NAFNetModel(
    NAFNet(
        image_shape=(3,256,256),
        base_channels=32,
        enc_blocks=[1,2,3,4],
        dec_blocks=[2,2,2,1],
        middle_blocks=6
    )
)

naf_history = naf.train_model(
    n_epochs=50,
    train_dataset=train_dataset,
    validation_dataset=validation_dataset,
    train_degradation=ImageDegradation(
        DegradationParameters(
            image_size=256,
            kernel_type="motion",
            kernel_size=9,
            motion_angle=45,
            noise_levels=[0.005, 0.01, 0.05, 0.1]
        )
    ),
    validation_degradation=ImageDegradation(
        DegradationParameters(
            image_size=256,
            kernel_type="motion",
            kernel_size=9,
            motion_angle=45,
            noise_levels=[0.01]
        )
    ),
    batch_size=8,
    learning_rate=1e-4,
    checkpoint_path="./weights/NAFNet/NAF_checkpoint.pth",
    resume=False,
    preview_every=5,
    preview_n=1
)

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/36 [00:00<?, ?it/s]

Epoch 1/50:   0%|          | 0/1250 [00:00<?, ?it/s]

KeyboardInterrupt: 